In [16]:
import profilelib.*
%use dataframe


In [17]:
@DataSchema
data class BenchmarkRow(
    val benchmark: String,
    val jvm: Double,
    val jvmError: Double,
    val native: Double,
    val nativeError: Double
)

fun loadAsDF(path: String): DataFrame<BenchmarkRow> {
    val jvm = loadScores(loadJson(path.replace("*", "jvm")))
    val native = loadScores(loadJson(path.replace("*", "linuxX64")))
    if (jvm.keys != native.keys) throw Exception()
    val benchmarks = jvm.keys.toList()

    fun Map<String, Double>.toNamedColumn(name: String): DataColumn<Double> = benchmarks
        .map { this[it]!! }
        .toColumn(name)

    return dataFrameOf(
        benchmarks.toColumn("benchmark"),
        jvm.mapValues { it.value.first }.toNamedColumn("jvm"),
        jvm.mapValues { it.value.second }.toNamedColumn("jvmError"),
        native.mapValues { it.value.first }.toNamedColumn("native"),
        native.mapValues { it.value.second }.toNamedColumn("nativeError"),
    ) as DataFrame<BenchmarkRow>
}

In [18]:
// Workaround for dataFrameConfig.display.decimalFormat = RendererDecimalFormat.of("%.2e")
// Confirmed by Kotlin Notebook that this doesn't and shouldn't work

fun DataFrame<*>.display() = convert { colsAtAnyDepth().colsOf<Double>() }.with { "%.2e".format(it) }

In [19]:
// f593cea Map benchmark

loadAsDF("../../benchmarks/vanilla/report-map-*.json")
    .add("ratio") { native / jvm }

benchmark,jvm,jvmError,native,nativeError,ratio
"MapBenchmark.controlB{size=""10""}",759119857.326849,2139402.725089,45320236.612710,207877.330196,0.059701
"MapBenchmark.controlB{size=""50""}",762446547.156493,18687427.474287,44668290.257533,193013.894885,0.058585
"MapBenchmark.controlB{size=""100""}",763925819.198096,34628628.802514,45450557.872337,72973.889414,0.059496
"MapBenchmark.controlB{size=""500""}",767693666.829037,29713713.731092,45286895.911187,73364.443041,0.058991
"MapBenchmark.controlB{size=""1000""}",757783143.131827,23556969.276731,44695049.715614,112422.409830,0.058981
"MapBenchmark.controlB{size=""5000""}",767072141.019164,29006489.952450,44836739.441361,381563.612162,0.058452
"MapBenchmark.controlB{size=""10000""}",748218820.942364,20046888.607717,46863722.060992,79497.341395,0.062634
"MapBenchmark.controlB{size=""50000""}",761032975.655372,11699539.865571,46695816.083055,62971.424581,0.061358
"MapBenchmark.controlB{size=""100000""}",749865409.047599,4897603.677021,46687021.726093,44141.337202,0.062261
"MapBenchmark.controlF{size=""10""}",746372402.644470,11452266.755355,45220468.578308,30572.529146,0.060587


In [20]:
// [master d719de9] Fix benchmarks for StringBuilder
loadAsDF("../../benchmarks/vanilla/report-stringbuilder-*.json")
    .add("ratio") { native / jvm }

benchmark,jvm,jvmError,native,nativeError,ratio
StringBuilderBenchmark.appendBig,2011.418537,10.703042,251.584028,2.408974,0.125078
StringBuilderBenchmark.appendMedium,1190.032920,13.736790,197.422228,1.986424,0.165896
StringBuilderBenchmark.appendSmall,191.171800,1.230805,19.300317,0.060918,0.100958


Let's do it with avgt instead.

In [21]:
fun findControlName(bmName: String): String = when (bmName.substringBefore(".")) {
    "MapBenchmark" -> bmName.takeIf { it.contains("get") }?.replace(Regex("get(.)\\d*")) { "control${it.groupValues[1]}" } ?: bmName
    "StringSubstringBenchmark" -> bmName.substringBefore("_") + "_control"
    "StringBuilderBenchmark" -> "StringBuilderBenchmark.control"
    else -> throw Exception()
}

// 491d85e Add control to StringBuilder

val df = loadAsDF("../../benchmarks/vanilla/report-avgt-*.json")
    .add("jvmControl") { df().single{ benchmark == findControlName(this@add.benchmark) }.jvm }
    .add("jvmControlError") { df().single{ benchmark == findControlName(this@add.benchmark) }.jvmError }
    .add("nativeControl") { df().single{ benchmark == findControlName(this@add.benchmark) }.native }
    .add("nativeControlError") { df().single{ benchmark == findControlName(this@add.benchmark) }.nativeError }
    .also { notebook.display(it.display()) }
    .filter { !benchmark.contains("control") }

benchmark,jvm,jvmError,native,nativeError,jvmControl,jvmControlError,nativeControl,nativeControlError
"MapBenchmark.controlB{size=""10""}",1.32e-09,1.03e-10,2.14e-08,4.65e-11,1.32e-09,1.03e-10,2.14e-08,4.65e-11
"MapBenchmark.controlB{size=""50""}",1.36e-09,5.87e-11,2.23e-08,1.03e-10,1.36e-09,5.87e-11,2.23e-08,1.03e-10
"MapBenchmark.controlB{size=""100""}",1.32e-09,5.60e-11,2.13e-08,3.25e-11,1.32e-09,5.60e-11,2.13e-08,3.25e-11
"MapBenchmark.controlB{size=""500""}",1.32e-09,2.58e-11,2.14e-08,6.66e-11,1.32e-09,2.58e-11,2.14e-08,6.66e-11
"MapBenchmark.controlB{size=""1000""}",1.37e-09,6.27e-11,2.14e-08,8.26e-11,1.37e-09,6.27e-11,2.14e-08,8.26e-11
"MapBenchmark.controlB{size=""5000""}",1.36e-09,2.94e-11,2.13e-08,6.94e-11,1.36e-09,2.94e-11,2.13e-08,6.94e-11
"MapBenchmark.controlB{size=""10000""}",1.32e-09,3.77e-11,2.24e-08,9.89e-11,1.32e-09,3.77e-11,2.24e-08,9.89e-11
"MapBenchmark.controlB{size=""50000""}",1.35e-09,7.62e-11,2.13e-08,8.92e-11,1.35e-09,7.62e-11,2.13e-08,8.92e-11
"MapBenchmark.controlB{size=""100000""}",1.35e-09,9.78e-11,2.13e-08,4.38e-11,1.35e-09,9.78e-11,2.13e-08,4.38e-11
"MapBenchmark.controlF{size=""10""}",1.35e-09,3.61e-11,2.23e-08,4.80e-11,1.35e-09,3.61e-11,2.23e-08,4.80e-11


In [22]:
df
    .add("naiveRatio") { jvm / native }
    .add("naiveRatioError") { propagateErrorThroughRatio(jvm, jvmError, native, nativeError).let { "%.2e".format(it) } }
    .add("adjustedRatio") { (jvm - jvmControl) / (native - nativeControl) }
    .add("adjustedRatioError") { propagateErrorThroughRatio(
        jvm - jvmControl,
        propagateErrorThroughSum(jvmError, jvmControlError),
        native - nativeControl,
        propagateErrorThroughSum(nativeError, nativeControlError),
    ).let { "%.2e".format(it) }}
    .remove { cols { (it.name() in df.columns().map { it.name }) && it.name() != "benchmark" } }


benchmark,naiveRatio,naiveRatioError,adjustedRatio,adjustedRatioError
"MapBenchmark.getB2{size=""10""}",0.037988,5.53e-07,0.022219,9.14e-07
"MapBenchmark.getB2{size=""50""}",0.039065,8.67e-08,0.022520,3.65e-07
"MapBenchmark.getB2{size=""100""}",0.038950,9.40e-08,0.022856,3.37e-07
"MapBenchmark.getB2{size=""500""}",0.038737,5.78e-08,0.023347,1.66e-07
"MapBenchmark.getB2{size=""1000""}",0.040112,5.69e-07,0.024169,8.12e-07
"MapBenchmark.getB2{size=""5000""}",0.039001,4.89e-07,0.021667,6.57e-07
"MapBenchmark.getB2{size=""10000""}",0.039006,5.90e-08,0.023620,2.47e-07
"MapBenchmark.getB2{size=""50000""}",0.038368,4.81e-07,0.021769,7.51e-07
"MapBenchmark.getB2{size=""100000""}",0.039928,9.77e-08,0.023563,5.70e-07
"MapBenchmark.getB5{size=""10""}",0.039892,5.06e-07,0.024205,8.91e-07
